# Question 8: LILA access and food-shelf site presence

Exploratory analysis of USDA LILA flags and food-shelf-type organizations across Minnesota's 2020 census tracts. This notebook creates statewide and Twin Cities tract maps and summarizes how sites are distributed across LILA and non-LILA tracts.

**Interpretation limit:** a site in a tract does not mean every resident can reach it. These are tract-level presence summaries, not population-weighted distance coverage or measures of site capacity, hours, or eligibility. The crosswalked Atlas flags are 2019-era data on 2010 tract geography.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch
from shapely.geometry import box

# Find the repository root so the notebook works from the repo or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find repository root containing src/config.py")
sys.path.insert(0, str(ROOT / "src"))
import config as C

BOUNDARIES = (C.EXTERNAL / "census" / "tract" / "tl_2020_27_tract"
              / "tl_2020_27_tract.shp")
LILA_DATA = C.PROCESSED / "lila_tracts_2020.parquet"
LILA_FLAG = "LILATracts_halfAnd10"
STATEWIDE_OUTPUT = C.FIG / "08_q8_lila_site_presence.png"
TWIN_CITIES_OUTPUT = C.FIG / "08_q8_lila_site_presence_twin_cities.png"
TWIN_CITIES_BOUNDS = (-94.15, 44.45, -92.45, 45.45)

for path in (BOUNDARIES, LILA_DATA, C.P_SITES):
    if not path.exists():
        raise FileNotFoundError(f"Required Q8 input not found: {path}")

## Load tract flags, site locations, and boundaries

ZIP-centroid geocodes are excluded. The `AgencyType` filter follows the project's shared `FOOD_SHELF_TYPES` definition. Site locations are summarized by their assigned tract; they are not plotted as address-level points.

In [ ]:
tracts = gpd.read_file(BOUNDARIES)
if "GEOID" not in tracts.columns:
    raise ValueError("TIGER/Line shapefile is missing its GEOID tract key")
tracts["tract_geoid"] = tracts["GEOID"].astype(str).str.zfill(11)

lila = pd.read_parquet(LILA_DATA)
if LILA_FLAG not in lila.columns:
    raise ValueError(f"Crosswalk is missing required flag: {LILA_FLAG}")
if lila["tract_geoid"].duplicated().any():
    raise ValueError("Crosswalk contains duplicate 2020 tract GEOIDs")

sites = pd.read_parquet(C.P_SITES)
placeable = sites[
    sites["state_fips"].astype(str).eq(C.STATE_FIPS_MN)
    & sites["has_geoid"].fillna(False).astype(bool)
    & ~sites["geoid_is_zip_centroid"].fillna(False).astype(bool)
    & sites["AgencyType"].isin(C.FOOD_SHELF_TYPES)
].copy()

site_counts = (placeable.groupby("tract_geoid")["site_id"]
               .nunique().rename("site_count").reset_index())
map_data = tracts.merge(lila[["tract_geoid", LILA_FLAG]], on="tract_geoid",
                        how="left", validate="one_to_one")
map_data = map_data.merge(site_counts, on="tract_geoid", how="left",
                          validate="one_to_one")
map_data["site_count"] = map_data["site_count"].fillna(0).astype(int)
map_data["has_site"] = map_data["site_count"].gt(0)

if map_data.crs is None:
    raise ValueError("TIGER/Line tract file has no coordinate reference system")
map_data = map_data.to_crs("EPSG:26915")
print(f"Boundary tracts: {len(tracts):,}")
print(f"Placeable food-shelf-type sites: {len(placeable):,}")
print(f"Tracts with at least one such site: {int(map_data['has_site'].sum()):,}")
print(f"Tracts with unavailable LILA status: {int(map_data[LILA_FLAG].isna().sum()):,}")

## Exploratory summaries

These tables describe tract and organization counts. They do **not** estimate the share of people served or within reach.

In [ ]:
map_data["lila_class"] = map_data[LILA_FLAG].map(
    {True: "LILA", False: "Non-LILA"}
).fillna("Unavailable")

tract_summary = (map_data.groupby("lila_class", observed=True)
                 .agg(tracts=("tract_geoid", "nunique"),
                      tracts_with_site=("has_site", "sum"),
                      site_organizations=("site_count", "sum"),
                      median_sites_per_tract=("site_count", "median")))
tract_summary["pct_tracts_with_site"] = (
    100 * tract_summary["tracts_with_site"] / tract_summary["tracts"]
)
tract_summary

In [ ]:
# Inspect how much tract-boundary ambiguity remains after the crosswalk.
crosswalk = pd.read_parquet(LILA_DATA)
parent_summary = (crosswalk.groupby("parent_count", as_index=False)
                  .agg(tracts=("tract_geoid", "size")))
parent_summary["pct_2020_tracts"] = (
    100 * parent_summary["tracts"] / len(crosswalk)
)
parent_summary

The crosswalk summary describes how many 2010 parent tracts contributed to each 2020 tract. The LILA share fields in `lila_tracts_2020.parquet` preserve the weighted values for inspecting borderline assignments.

In [ ]:
COLORS = {
    "LILA tract with site": "#238b8e",
    "LILA tract without site": "#e76f51",
    "Non-LILA tract with site": "#82a876",
    "Non-LILA tract without site": "#edf0f2",
    "LILA status unavailable": "#b8b8b8",
}
known_lila = map_data[LILA_FLAG].notna()
is_lila = map_data[LILA_FLAG].astype("boolean").fillna(False)
map_data["map_category"] = "LILA status unavailable"
map_data.loc[known_lila & is_lila & map_data["has_site"], "map_category"] = "LILA tract with site"
map_data.loc[known_lila & is_lila & ~map_data["has_site"], "map_category"] = "LILA tract without site"
map_data.loc[known_lila & ~is_lila & map_data["has_site"], "map_category"] = "Non-LILA tract with site"
map_data.loc[known_lila & ~is_lila & ~map_data["has_site"], "map_category"] = "Non-LILA tract without site"

def draw_map(data, title, extent=None):
    fig, ax = plt.subplots(figsize=(10, 9) if extent else (10, 10))
    for category, color in COLORS.items():
        subset = data[data["map_category"].eq(category)]
        if not subset.empty:
            subset.plot(ax=ax, color=color,
                        linewidth=0.25 if extent else 0.12,
                        edgecolor="white")
    if extent:
        projected_extent = gpd.GeoSeries(
            [box(extent[0], extent[1], extent[2], extent[3])],
            crs="EPSG:4326").to_crs(data.crs)
        xmin, ymin, xmax, ymax = projected_extent.total_bounds
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
    fig.suptitle(title, x=0.5, y=0.97, fontsize=15, fontweight="bold")
    fig.text(0.5, 0.935,
             "2020 tract boundaries | USDA 2019 half-mile/10-mile LILA definition",
             ha="center", fontsize=10, color="#444444")
    fig.legend(handles=[Patch(facecolor=color, edgecolor="white", label=label)
                        for label, color in COLORS.items()],
               title="Tract classification", loc="lower center",
               bbox_to_anchor=(0.5, 0.015), ncol=2,
               frameon=True, framealpha=0.95)
    fig.text(0.5, 0.105,
             "Site presence is assigned to the site's tract, not an exact address or service radius.",
             ha="center", fontsize=8, color="#444444")
    ax.set_axis_off()
    fig.subplots_adjust(top=0.91, bottom=0.15)
    fig.savefig(output, dpi=220)
    plt.show()
    plt.close(fig)

In [ ]:
output = STATEWIDE_OUTPUT
draw_map(map_data, "Minnesota food shelves and low-income, low-access tracts")
print(f"Saved {output.relative_to(ROOT)}")

In [ ]:
output = TWIN_CITIES_OUTPUT
draw_map(map_data,
         "Twin Cities food shelves and low-income, low-access tracts",
         extent=TWIN_CITIES_BOUNDS)
print(f"Saved {output.relative_to(ROOT)}")

## Takeaways and next questions

- Compare the percentage of LILA and non-LILA tracts containing at least one food-shelf-type organization. This is tract presence only; do not describe it as population coverage.
- Use the Twin Cities view to inspect local clustering, while remembering that tract size varies and this map does not show residents' distance to a site.
- Next quantify population-weighted nearest-site distance and compare food-shelf locations with grocery-store access. Those analyses are needed to answer whether the network reaches people in LILA areas or fills retail gaps.

All maps use the same tract classifications. Source inputs are the processed site master, the 2020 LILA crosswalk, and the 2020 Minnesota TIGER/Line tract boundaries.